In [40]:
import numpy as np
import matplotlib.pyplot as plt

# Matrix Operations

### Determinant of a matrix

The determinant of a square matrix is a scalar value that can be computed from its elements. It provides important information about the matrix, such as whether the matrix is non-singular (non-zero determinant) and the volume scaling factor of the linear transformation represented by the matrix.

For a 2×2 matrix:
$$ \det\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc $$

For a 3×3 matrix:
$$ \det\begin{pmatrix} a & b & c \\ d & e & f \\ g & h & i \end{pmatrix} = a(ei - fh) - b(di - fg) + c(dh - eg) $$

The determinant has several important properties:
- If determinant of a matrix is 0,  the matrix is singular.
- If determinant of a matrix is not 0, the matrix is non-singular.

In [41]:
mat = np.array([
    [-1, 3],
    [3, 2]
])
print(mat)
determinant = np.linalg.det(mat)
print("determinant: ", determinant)

[[-1  3]
 [ 3  2]]
determinant:  -11.000000000000002


### Row Echelon Form (REF)

A matrix is in **Row Echelon Form** when:
- All zero rows are at the bottom.
- The leading entry (first non-zero number) of each non-zero row is to the **right** of the leading entry of the row above it.
- All entries **below** a leading entry are zero.

A matrix is in **Reduced Row Echelon Form (RREF)** when, additionally:
- Each leading entry is **1** (called a pivot).
- All entries **above and below** each pivot are zero.

Row operations used to achieve REF:
1. **Swap** two rows.
2. **Scale** a row by a non-zero scalar.
3. **Add** a multiple of one row to another.

**Example:**

Starting matrix:
$$\begin{pmatrix} 1 & 2 & -1 & 8 \\ 2 & 4 & 1 & 17 \\ -1 & 2 & 5 & 4 \end{pmatrix}$$

After Row Echelon reduction:
$$\begin{pmatrix} 1 & 2 & -1 & 8 \\ 0 & 1 & 1.5 & 0.5 \\ 0 & 0 & 1 & -1 \end{pmatrix}$$

> **Note:** The code block below implements Gaussian elimination to convert an integer matrix into Row Echelon Form, printing the original and the transformed result.

In [42]:
# Example: Convert a matrix to Row Echelon Form (REF) using Gaussian elimination
matrix = np.array([
    [1, 2, -1, 8],
    [2, 4, 1, 17],
    [-1, 2, 5, 4]
], dtype=int)

print("Original matrix:")
print(matrix)

def row_echelon(matrix):
    m = matrix.astype(float).copy()
    rows, cols = m.shape
    pivot_row = 0

    for col in range(cols):
        if pivot_row >= rows:
            break

        # Find a pivot in current column
        pivot = None
        for r in range(pivot_row, rows):
            if abs(m[r, col]) > 1e-10:
                pivot = r
                break

        if pivot is None:
            continue

        # Swap to bring pivot to current pivot_row
        if pivot != pivot_row:
            m[[pivot_row, pivot]] = m[[pivot, pivot_row]]

        # Normalize pivot row (make leading entry 1)
        m[pivot_row] = m[pivot_row] / m[pivot_row, col]

        # Eliminate entries below pivot
        for r in range(pivot_row + 1, rows):
            m[r] = m[r] - m[r, col] * m[pivot_row]

        pivot_row += 1

    # Clean tiny floating-point noise
    m[np.abs(m) < 1e-10] = 0
    return m

ref = row_echelon(matrix)
print("Row Echelon Form (REF):")
print(ref)

Original matrix:
[[ 1  2 -1  8]
 [ 2  4  1 17]
 [-1  2  5  4]]
Row Echelon Form (REF):
[[ 1.          2.         -1.          8.        ]
 [ 0.          1.          1.          3.        ]
 [ 0.          0.          1.          0.33333333]]


### Reduced Row Echelon Form (RREF)
> **Note:** The code block below extends the REF function by also eliminating entries **above** each pivot, producing the fully reduced form where each pivot column has exactly one non-zero entry.

In [ ]:
# Example: Convert a matrix to Reduced Row Echelon Form (RREF)
# Uses a square coefficient-only matrix so the result contains only 0s and 1s
coeff_matrix = np.array([
    [1, 2, 3],
    [0, 1, 4],
    [5, 6, 0]
], dtype=int)

def reduced_row_echelon(matrix):
    m = matrix.astype(float).copy()
    rows, cols = m.shape
    pivot_row = 0

    for col in range(cols):
        if pivot_row >= rows:
            break

        # Find a pivot in the current column
        pivot = None
        for r in range(pivot_row, rows):
            if abs(m[r, col]) > 1e-10:
                pivot = r
                break

        if pivot is None:
            continue

        # Swap to bring pivot to current pivot_row
        if pivot != pivot_row:
            m[[pivot_row, pivot]] = m[[pivot, pivot_row]]

        # Normalize pivot row so leading entry becomes 1
        m[pivot_row] = m[pivot_row] / m[pivot_row, col]

        # Eliminate entries both above and below the pivot
        for r in range(rows):
            if r != pivot_row:
                m[r] = m[r] - m[r, col] * m[pivot_row]

        pivot_row += 1

    m[np.abs(m) < 1e-10] = 0
    return m.astype(int)

rref = reduced_row_echelon(coeff_matrix)
print("Original matrix:")
print(coeff_matrix)
print("Reduced Row Echelon Form (RREF):")
print(rref)

Original matrix:
[[1 2 3]
 [0 1 4]
 [5 6 0]]

Reduced Row Echelon Form (RREF):
[[1 0 0]
 [0 1 0]
 [0 0 1]]


### Rank of a Matrix using Reduced Row Echelon Form

The **rank** of a matrix is the number of **non-zero rows** in its Reduced Row Echelon Form (RREF).

- Each non-zero row in the RREF corresponds to a **linearly independent row** of the original matrix.
- A zero row in the RREF means those equations were **redundant** (dependent on others).
- For an $m \times n$ matrix: $\text{rank}(A) \leq \min(m, n)$
- If $\text{rank}(A) = n$ (number of columns), the matrix has **full column rank** (columns are independent).
- If $\text{rank}(A) = m$ (number of rows), the matrix has **full row rank** (rows are independent).

> **Note:** The code block below computes the RREF using `reduced_row_echelon`, then counts the non-zero rows to determine the rank, and cross-verifies with `np.linalg.matrix_rank`.

In [44]:
def rank_via_rref(matrix):
    # Step 1: Get the RREF
    rref = reduced_row_echelon(matrix)
    print("RREF:")
    print(rref)

    # Step 2: Count non-zero rows — each represents a linearly independent row
    rank = 0
    for row in rref:
        if np.any(row != 0):
            rank += 1
    return rank

def print_rank_and_determinant(matrix):
    rows, cols = matrix.shape
    rank = rank_via_rref(matrix)
    print("Rank :", rank)

    # Determinant is only defined for square matrices
    # Relation: rank == n (full rank) => det != 0 (non-singular)
    #           rank <  n             => det == 0  (singular)
    if rows == cols:
        det = np.linalg.det(matrix)
        print("Determinant :", det)
        if rank == rows:
            print("Full rank => matrix is invertible (det != 0)")
        else:
            print("Rank deficient => matrix is singular (det == 0)")
    else:
        print("Determinant : not defined (non-square matrix)")

# Test with a full-rank matrix
matrix = np.array([
    [1, 2, 3],
    [0, 1, 4],
    [5, 6, 0]
], dtype=int)

print("Matrix:")
print(matrix)
print()
print_rank_and_determinant(matrix)

# Test with a rank-deficient matrix (row 3 = row 1 + row 2)
matrix = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [5, 7, 9]
], dtype=int)

print("")
print("Rank-deficient matrix:")
print(matrix)
print()
print_rank_and_determinant(matrix)

Matrix:
[[1 2 3]
 [0 1 4]
 [5 6 0]]

RREF:
[[1 0 0]
 [0 1 0]
 [0 0 1]]
Rank : 3
Determinant : 0.9999999999999964
Full rank => matrix is invertible (det != 0)

Rank-deficient matrix:
[[1 2 3]
 [4 5 6]
 [5 7 9]]

RREF:
[[ 1  0 -1]
 [ 0  1  2]
 [ 0  0  0]]
Rank : 2
Determinant : 2.664535259100367e-15
Rank deficient => matrix is singular (det == 0)
